# bachelor-3d: GPU na Kaggle

Nastavenia notebooku (pravý panel → Session options):
- **Accelerator: GPU T4 x2** (nie P100 — staršia architektúra sm_60),
- **Internet: On** (vyžaduje overený telefón v účte).

Výstupy v `/kaggle/working` sa uchovajú iba po **Save Version** (aj beh na pozadí: *Save & Run All*).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

## 1. Repozitár a závislosti

In [ ]:
import os
REPO = '/kaggle/temp/bachelor-3d'   # mimo /kaggle/working, aby repozitár nebol vo výstupe
LOGS = '/kaggle/working/logs'
os.makedirs(LOGS, exist_ok=True)
if not os.path.isdir(REPO):
    !git clone https://github.com/UladzislauHrynevich/bachelor-3d.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!pip install -q gsplat==1.5.3

## 2. Smoke test (rasterizácia + backward + 200 krokov)

Prvé volanie kompiluje CUDA jadrá (~12 min na T4). Log ide do `/kaggle/working/logs/`.

In [ ]:
import datetime
LOG = f"{LOGS}/{datetime.datetime.now():%Y%m%d-%H%M}-smoke.log"
!MAX_JOBS=4 python gpu/smoke_test.py 2>&1 | tee "{LOG}"
print('log:', LOG)

## 3. Referenčná rekonštrukcia COLMAP (pycolmap, CPU)

Vstup: dataset `south-building` (Add Input). Jedna spoločná kamera, ostatné parametre predvolené.
Výstup: `/kaggle/working/colmap_reference/` (database.db, sparse/0, colmap_run.json) — uložiť cez **Save Version**.

In [ ]:
!pip install -q pycolmap==4.2.1
import glob
IMAGES = '/kaggle/input/south-building/images'   # upraviť, ak má dataset iný názov
print(len(glob.glob(f'{IMAGES}/*.JPG')), 'snímok')

In [ ]:
import datetime  # bunku smoke testu možno preskočiť
LOG = f"{LOGS}/{datetime.datetime.now():%Y%m%d-%H%M}-colmap.log"
!python gpu/colmap_reference.py {IMAGES} /kaggle/working/colmap_reference 4 2>&1 | grep -v '^[IW][0-9]' | tee "{LOG}"
print('log:', LOG)